# МТС · ресурсный пилот +500 / FP16–INT8–INT4

**Только измерения, без обучения.** Все режимы получают одинаковые4 звонка и17 gold-фактов, однаGPU, batch1, greedy/KV-cache, один проход. Нормализация tariff-catalog-v1 совпадает с текущим приложением. Размер логических параметров при квантовании не уменьшается. Это диагностические точки на малой выборке, не итоговое качество383.

Запускать на свободнойGPU: в другой VM участника команды либо после завершения383. Не запускать одновременно с383 на той жеGPU. ДляCPU предусмотрен отдельный локальный START_CPU_BENCHMARK.cmd — GPU этого ноутбука ему не нужна.


## 1. GPU, Drive, папка результата
ПредпочтительноA100/L4. НазваниеGPU фиксируется автоматически.


In [ ]:
import json,time,sys,subprocess,os,hashlib,zipfile,re
from pathlib import Path
from datetime import datetime,timezone
from google.colab import drive,files
subprocess.run(['nvidia-smi'],check=True)
drive.mount('/content/drive')
RESUME_RUN=''
RUNS=Path('/content/drive/MyDrive/MTS_Qwen06_Resources');RUNS.mkdir(exist_ok=True)
locator=RUNS/'last_gpu_run.json'
if RESUME_RUN:RUN=Path(RESUME_RUN)
elif locator.exists():RUN=Path(json.loads(locator.read_text())['run'])
else:RUN=RUNS/datetime.now(timezone.utc).strftime('gpu_%Y%m%dT%H%M%SZ')
RUN.mkdir(parents=True,exist_ok=True);locator.write_text(json.dumps({'run':str(RUN)}))
origin_path=RUN/'origin.json'
if origin_path.exists():ORIGIN=json.loads(origin_path.read_text())['origin_epoch']
else:ORIGIN=time.time();origin_path.write_text(json.dumps({'origin_epoch':ORIGIN}))
print('Результаты:',RUN,'Бюджет: до60минут с этой ячейки, не позже07:15МСК')


## 2. Загрузить ZIP и установить библиотеки
Нужен только mts_qwen06_resources.zip. Torch/CUDA не переустанавливаются.


In [ ]:
payload=Path('/content/mts_qwen06_resources.zip')
EXPECTED='194a67c42afee618ba413c490e67844be7cc88f80a9c5176c5de3dbd21e6d8ad'
if not payload.exists() or hashlib.sha256(payload.read_bytes()).hexdigest()!=EXPECTED:
    uploaded=files.upload()
    found=False
    for name,data in uploaded.items():
        canonical=re.sub(r' \(\d+\)(?=\.zip$)','',Path(name).name)
        if canonical=='mts_qwen06_resources.zip':
            assert hashlib.sha256(data).hexdigest()==EXPECTED,'Другая версия ZIP'
            payload.write_bytes(data);found=True
    del uploaded
    assert found,'Нужен mts_qwen06_resources.zip'
assert hashlib.sha256(payload.read_bytes()).hexdigest()==EXPECTED
ROOT=Path('/content/mts_qwen06_resources');ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(payload) as z:
    assert len(z.namelist())==len(set(z.namelist())) and sum(i.file_size for i in z.infolist())<200*1024**2
    assert all((ROOT/n).resolve().is_relative_to(ROOT.resolve()) and chr(92) not in n and ':' not in n for n in z.namelist())
    assert z.testzip() is None
    z.extractall(ROOT)
m=json.loads((ROOT/'package_manifest.json').read_text())
for rel,want in m['sha256'].items():
    p=(ROOT/rel).resolve();assert p.is_relative_to(ROOT.resolve()) and hashlib.sha256(p.read_bytes()).hexdigest()==want,rel
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements_gpu.txt')],check=True)
print('Готовы4 звонка и один проверенный адаптер +500. Обучение не запускается.')


## 3. Измерить три режима и скачать результат
После загрузки каждого режима появятся START/DONE1/4…4/4. Если INT8/INT4 не загрузятся, они будут помечены not_measured, не выдуманными точками. Сохранённые записи повторно не генерируются в том жеRUN.


In [ ]:
env=os.environ.copy();env['PYTHONPATH']=str(ROOT);env['TOKENIZERS_PARALLELISM']='false';env['PYTHONUNBUFFERED']='1'
env['BENCH_ORIGIN_EPOCH']=str(ORIGIN);env['HF_HUB_OFFLINE']='0';env['TRANSFORMERS_OFFLINE']='0'
env.pop('MTS_MODEL_BASE_PATH',None)
remaining=min(ORIGIN+3600,datetime.fromisoformat('2026-11-16T07:15:00+03:00').timestamp())-time.time()
try:
    assert remaining>60,'Бюджет истёк; автоматически не сбрасываем'
    result=subprocess.run([sys.executable,'-u',str(ROOT/'resource_benchmark.py'),'--root',str(ROOT),'--out',str(RUN),
        '--device','cuda','--modes','fp16','int8','int4','--budget-minutes','60'],cwd=ROOT,env=env,timeout=remaining)
    print('Код завершения:',result.returncode)
except subprocess.TimeoutExpired:print('Достигнут лимит времени; частичные записи сохранены.')
finally:
    if (RUN/'summary.json').exists():
        try:subprocess.run([sys.executable,str(ROOT/'plot_resources.py'),str(RUN)],cwd=ROOT,env=env,check=True)
        except subprocess.CalledProcessError:print('График не создан; исходные измерения сохранены.')
    subprocess.run([sys.executable,str(ROOT/'pack_run.py'),str(RUN)],cwd=ROOT,check=True)
    files.download(str(RUN.parent/(RUN.name+'_results.zip')))


## 4. Проверить завершённость
Для трёх точек нужен all_three_gpu_points_complete=true. Ошибки генерации остаются в отчёте. Полный val383 и ресурсное сравнение с большим baseline этим пилотом не заменяются.


In [ ]:
for name in ['summary.json','failure.json']:
    p=RUN/name
    if p.exists():print(name,p.read_text(encoding='utf-8'))
